# Ensemble Methods - Practical Overview

## First Part - Necessary Code

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from auxiliary_functions import chart

In [2]:
# Import company colors
with open('plot_colors.json', 'r') as pc:
    color_dict = json.load(pc)

c_light, c_dark, c_blue, theme_flowchart = color_dict['color_light'], color_dict['color_dark'], color_dict['color_blue'], color_dict['theme_flowchart']

# Define color map for plotting
#color_map = ListedColormap([c_blue, c_light, c_dark])

In [3]:
# Generate Data for plots
np.random.seed(42)
X = np.random.rand(100, 1) - 0.5
y = 3*X[:, 0]**2 + 0.05 * np.random.randn(100)

In [4]:
# Train model with 3 estimators and learning rate 1
gbrt_3 = GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0, random_state=42)
gbrt_3.fit(X, y)

# Train model with 200 estimators and learning rate 0.1
gbrt_200 = GradientBoostingRegressor(max_depth=2, n_estimators=200, learning_rate=0.1, random_state=42)
gbrt_200.fit(X, y)


def plot_ensemble(X, y, models, figsize=(15,5)):
    
    axis_limit = [-.5, 0.5, -0.1, 0.8]
    
    x1 = np.linspace(axis_limit[0], axis_limit[1], 500)
    
    y_pred_3 = models[0].predict(x1.reshape(-1, 1))
    y_pred_200 = models[1].predict(x1.reshape(-1, 1))
    
    fig, ax = plt.subplots(1, 2, figsize=figsize, sharey=True)
    sns.set_context("talk", font_scale=0.8)
    
    ax[0].scatter(X, y, c=c_dark, s=20)
    ax[0].plot(x1, y_pred_3, c=c_light, linewidth=2.5, label="Ensemble predictions")
    ax[0].set(xlabel='X', ylabel='y', title='learning_rate=1.0, n_estimators=3')
    ax[0].axis(axis_limit)
    
    ax[1].scatter(X, y, c=c_dark,  s=20)
    ax[1].plot(x1, y_pred_200, c=c_light, linewidth=2.5, label="Ensemble predictions")
    ax[1].set(xlabel='X', title='learning_rate=0.1, n_estimators=200')
    ax[0].axis(axis_limit)
    
    ax[0].legend()
    ax[1].legend()

In [5]:
# Train test split
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=49)

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120, random_state=42)
gbrt.fit(X_train, y_train)

errors = [mean_squared_error(y_val, y_pred) for y_pred in gbrt.staged_predict(X_val)]
bst_n_estimators = np.argmin(errors) + 1

gbrt_best = GradientBoostingRegressor(max_depth=2, n_estimators=bst_n_estimators, random_state=42)
gbrt_best.fit(X_train, y_train)

min_error = np.min(errors)


def plot_early_stopping(X, y, model, figsize=(15,5)):
    
    axis_limit = [-.5, 0.5, -0.1, 0.8]
    
    x1 = np.linspace(axis_limit[0], axis_limit[1], 500)
    
    y_pred_best = model.predict(x1.reshape(-1, 1))
    
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    sns.set_context("talk", font_scale=0.8)
    
    ax[0].plot(errors, c=c_blue, marker='o', markersize=5)
    ax[0].plot([bst_n_estimators, bst_n_estimators], [0, min_error], "k--")
    ax[0].plot([0, 120], [min_error, min_error], "k--")
    ax[0].text(gbrt_best.n_estimators_, min_error*1.2, "Minimum", ha="center")
    ax[0].set(xlabel='Number of trees', ylabel='Error', title='Validation Error')
    ax[0].axis([0, 120, 0, 0.01])
    
    ax[1].scatter(X, y, c=c_dark,  s=20)
    ax[1].plot(x1, y_pred_best, c=c_light, linewidth=2.5)
    ax[1].set(xlabel='X', ylabel="y", title=f"Best model ({gbrt_best.n_estimators_} trees)")
    ax[1].axis(axis_limit)

## Second Part - Lecture

<div class="slide-title">

# Ensemble Methods
    
</div>    

Notes:  
This session is adapted for the AIPM cohort.  
Focus: choosing the right ensemble approach for product and business problems, not deriving formulas.

## Agenda

1. Quick recap: why single trees are unstable
2. Bagging and Random Forests
3. Extra Trees and when to use them
4. Boosting family (AdaBoost, Gradient Boosting, XGBoost, LightGBM, CatBoost)
5. Stacking and blending
6. Calibration and interpretability
7. AIPM decision guide: choosing the right ensemble for your use case

Notes:  
"recap" in case random forests have been shown the day before.

## 1. Recap CART - Classification and regression trees

<div class="slide-chapter-split">
    <div class="title-chapter">Recap CART - Classification and regression trees</div>
    <div class="image-container">
        <img src="../../images/ensemble_methods/img_p2_1.png">
    </div>
</div>

Notes: CART is the algorithm used for the decision tree algorithm.

### CART (Classification and Regression Tree)

<div class="group">
    <div class="text_70">

**General:**

* Finding the globally best split sequence is computationally hard.
* CART uses a greedy strategy: best split now, then repeat.
* At each node, CART picks the **best feature** and **best threshold** for that local split.

**How split quality is measured:**

* Regression: pick splits that reduce average prediction error (MSE).
* Classification: pick splits that increase class purity (for example with Gini impurity).
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../../images/decision_tree/img_p13_1.png">
    </div>
</div> 

<div class="alert alert-block alert-info">
<b>Note:</b> 
NP-complete (nondeterministic polynomial-time complete) defines a problem which is easy to understand, easy to verify but difficult to solve e.g. a jigsaw puzzle.
</div>

Notes:  
For this session, we keep CART conceptual and avoid mathematical detail.  

- CART uses greedy splitting decisions at each node.
- For regression, split quality is based on prediction error reduction.
- For classification, split quality is based on class purity improvement.

NP-complete is a formal complexity term: a problem may be easy to verify, but hard to solve optimally.

### CART 

<div class="group">
    <div class="text_70">

**Regularisation:**

* pruning to avoid overfitting
* Grow a full tree and then prune e.g.
    - `max_leaves`
    - `max_depth`
    - `min_sample_size`
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../../images/decision_tree/img_p13_1.png">
    </div>
</div> 


### CART - pros and cons

<br>

<table class="confusion_matrix" border="1" width="1800">
  <thead>
    <tr>
      <th><b>&nbsp;&nbsp;PROS&nbsp;&nbsp;</b></th>
      <th><b>&nbsp;&nbsp;CONS&nbsp;&nbsp;</b></th>
    </tr>
  </thead>
</table>

### CART - pros and cons

<br>

<table class="confusion_matrix" border="1" width="1400">
  <thead>
    <tr>
      <th><b>PROS</b></th>
      <th><b>CONS</b></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td class="smaller-font-size">white box model - easy to interpret</td>
      <td class="smaller-font-size">not very accurate ...</td>
    </tr>
    <tr>
      <td class="smaller-font-size">can handle mixed data, discrete and continuous</td>
      <td class="smaller-font-size">greedy nature of constructing</td>
    </tr>
    <tr>
      <td class="smaller-font-size">insensitive to transformations of data... split points based on ranking  </td>
      <td class="smaller-font-size">trees are unstable, small changes to the input can lead to large changes in tree structure</td>
    </tr>
    <tr>
      <td class="smaller-font-size">relative robust to outliers</td>
      <td class="smaller-font-size">tend to overfit (aka high variance estimators)</td>
    </tr>
  </tbody>
</table>

### Trees being unstable

**Q:** Should you trust and follow a single CART?

<div class="group">
    <div class="text_70">       
        
* Small data changes can produce very different tree structures and decisions.
* As in investing, it is risky to put all eggs in one basket.
    </div>
    
    <div class="images_30">   
        <img src="../../images/ensemble_methods/img_p7_1.png">
    </div>
</div>

### 2. Trees being greedy and overfitting

<div class="group">
    <div class="text_70">

How to find out what to order at the restaurant?
* **Try everything!** <br>... and then order what you liked best.
* **Ask your friends to try something**, <br>find out which they preferred more and try those maybe?

**Q:** Which do you find more feasible?
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../../images/ensemble_methods/eating_cloud.png">
    </div>
</div> 


<center>
    <img src="../../images/ensemble_methods/img_p9_1.png" position="center" size="cover">
</center>

Notes:  
Who Wants to Be a Millionaire offers two kinds of help: an individual friend and the audience.  
Across many episodes, the audience is typically more accurate than a single friend. This is a classic crowd wisdom example: aggregated judgments can outperform individual judgments when the crowd is diverse and reasonably independent.

## Wisdom of Crowds Theory

* Diversity of opinions
* Independence of opinions
* Decentralisation
* Aggregation

Notes:  
Crowd wisdom works best when four conditions hold:  
1. People bring different information.
2. People form judgments independently.
3. Decision-making is decentralized.
4. A clear aggregation mechanism combines opinions.

Prompt for class discussion: do these conditions hold for both game-show audiences and platforms like Wikipedia?

<div class="slide-chapter">

## 3. How can we produce different models with the same data?

</div>


<div class="slide-chapter">

## I. Bagging

</div>

### Bagging

<span class="color-brand" style="font-weight:bold;">B</span>ootstrap <span class="color-brand" style="font-weight:bold;">agg</span>regat<span class="color-brand" style="font-weight:bold;">ing</span>

Train many models on different bootstrap samples, then combine predictions:

* Classification: majority vote
* Regression: average prediction

Training many **different trees** on **different subsets** of data chosen randomly with replacement.

Notes:  
mean (instead of mode) when we have a regression problem.

### Bootstrapping

<div class="group">
    <div class="text_70">

**Intuition:** Create many training samples from one dataset by...

...*sampling with replacement* 
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../../images/ensemble_methods/bootstrapping.png">
    </div>
</div>

Notes:  
1. In Random Forest, we sample observations with replacement for each tree.  
2. We also sample a subset of features to increase model diversity.  
3. Repeating this process creates many different trees.

Bootstrapping resamples one dataset into many simulated samples. Each sample trains one model, and predictions are aggregated by voting (classification) or averaging (regression).

Reference: https://statisticsbyjim.com/hypothesis-testing/bootstrapping/

## Majority Voting - how it works

For **each observation** you have **different predictions** from **different models**

You need one prediction... so you aggregate

Aggregating the different models can be done in many ways e.g. averages, mode, median




### Majority Voting - how it works

For each observation you have different predictions. How do you aggregate this to get one value?

### Hard Voting


<div class="group">
    <div class="text_70">
takes the majority  
class 
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../../images/ensemble_methods/hard_voting.png">
    </div>
</div> 




### Soft Voting


<div class="group">
    <div class="text_70">
average of summed probability 
vectors  
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../../images/ensemble_methods/hard_voting.png">
    </div>
</div> 




<div class="alert alert-block alert-info">
<b>Note:</b> 
    
Soft voting tends to outperform hard voting.

</div>

### Majority Voting - why it works

When models make different mistakes, combining them cancels out part of the noise.

Key idea:

- Averaging multiple predictions usually makes results more stable than relying on one model.
- The gain is strongest when models are both:
  - individually better than random
  - not too similar to each other
- If all models are highly correlated, the benefit drops.

Practical takeaway for AIPM:

- More models is not automatically better.
- Better diversity in models and data slices often matters more than raw model count.

Note:  
The central message is simple: averaging independent signals reduces volatility.  

To get a stable ensemble, we want:
1. low correlation between learners (diversity)
2. enough learners
3. learners that are at least slightly better than random

Remember: ensembling mainly reduces variance. It does not magically fix every bias problem.

A business-friendly way to explain this:

- Imagine asking many analysts for the same forecast.
- If each analyst has small independent errors, the average forecast is usually closer to reality.
- If all analysts copy each other, you lose that benefit.

So ensemble quality depends on two levers:

1. Competence of each learner
2. Diversity across learners

### Majority Voting - why it works  

Model error has three parts in practice:

- bias: model is too simple for the pattern
- variance: model is too sensitive to training noise
- irreducible noise: randomness in the real world

Bagging and Random Forests are especially useful for reducing variance.

Notes:  
Students might remember this Error Decomposition from Bias-Variance Tradeoff lesson.  
While the Bias is small for decision trees, the variance gets small by using many of them (RF). Unfortunately bootstrapping can lead to correlations between the trees.

### Majority Voting - why it works

<div class="group">
    <div class="text_70">

**Intuition:** not all classifiers fail on the same observation at the same time.

**Prerequisites:**
* models should be better than random (weak learners)
* models should be diverse (low correlation)
* you need a sufficient number of learners
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../../images/ensemble_methods/multible_learners.png">
    </div>
</div>

<div class="alert alert-block alert-info">
<b>Note:</b> 

In classification, a weak learner performs slightly better than random guessing.

</div>

Notes:  
This statement refers to weak classification learners.  
In context of regression weak learners: They should at least be slightly better than simple average or simple linear regression.  

### Majority Voting - why it works

<div class="group">
    <div class="text_70">

**What does model independence mean?**

If two weak learners are highly correlated, they tend to repeat the same mistakes.
    </div>
    <br>
    <div class="images_30">
        <img src="../../images/ensemble_methods/weak_learners.png">
        **<span class="color-brand">A weak learner is at least slightly better than random.</span>**
    </div>
</div>

<div class="alert alert-block alert-info">
<b>Note:</b> 

You can demonstrate this with simulation (for example, repeated coin-toss style experiments) without using formal derivations.

</div>

### Example of weak learners

<div class="group">
    <div class="text">

If each learner is correct only slightly more than half the time,
majority voting can still produce a strong ensemble.

Scenario:

- 1000 learners
- each learner: about 51 percent accurate

Even this small edge can produce a much better combined decision.
    </div>
    <div class="images">
        Cumulative distribution function (cdf)
        <img src="../../images/ensemble_methods/img_p21_1.png" width=400>
    </div>
</div> 

Notes:  
Think of each weak learner as a slightly biased coin (about 51 percent correct).  
With many independent learners, majority voting is much more accurate than any single learner.  
The core message: a small individual edge can become a strong collective edge.

### Example of weak learners

<div class="group">
    <div class="text">

If learner quality increases a bit more, ensemble reliability improves sharply.

Scenario:

- 1000 learners
- each learner: about 55 percent accurate

The chance that majority voting is wrong becomes very small.
    </div>
    <div class="images">
        Cumulative distribution function (cdf)
        <img src="../../images/ensemble_methods/img_p22_1.png" width=400>
    </div>
</div> 

Notes:  
Explain this slide as a simulation story:

- Each weak learner is slightly better than random.
- One learner alone is unreliable.
- A large independent group of such learners is much more reliable under majority voting.

Keep the focus on intuition, not probability math: independence + slight skill advantage + aggregation leads to strong ensemble performance.

### Reducing variance by aggregation

<center>
    <img src="../../images/ensemble_methods/img_p14_1.png" position="center">
</center>

Notes:  
This is the practical bridge to Random Forest: split data into different views, train many trees, and aggregate their outputs.  
The result is usually more stable and less sensitive to noise than a single tree.

### Random forest

Random Forest decorrelates trees by introducing randomness in both data and features.

* randomly chosen **subset of input variables (features)**
    * different features are considered at different nodes

* randomly chosen **subset of observations**
    * creates out-of-bag instances for validation

Random Forest usually improves predictive performance, but reduces interpretability versus a single tree.

Notes:  
Many base learners are possible, but decision trees are common because they are high-variance and benefit strongly from ensembling.  
Their sensitivity to training data also helps create diverse learners.

Terminology:
- sampling both features and observations: random patches
- sampling features only: random subspaces

Random Forest can also provide feature-importance estimates.

### Random Forest


<div class="group">
    <div class="text_70">
using several 
decision trees
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../../images/ensemble_methods/randomforest_concept.png">
    </div>
</div> 




<div class="alert alert-block alert-info">
<b>Note:</b> 
    
Running same learning algorithms on different subsets can result in still highly correlated predictors.

</div>

### Extra-Trees - Extremely Randomized Trees

Introduce even more randomness
* For each node, random split points are generated (rather than exhaustively testing all possibilities).  
 The splitting point with the best outcome is used for that node.  
* *All* the data in train is used to build each split (no bootstrapping).  
 (But like in random forest, a subset of random features is used.)  
* The trees are usually not pruned (similar to random forest).


Notes:  
(One common implementation approach.)
1. More random split selection can increase bias but reduce correlation and training cost.
2. Lower correlation improves the ensemble effect.
3. Overall, this can still lead to strong generalization.

In summary: ensembles reduce variance most effectively when base learners are diverse.

### Accuracy vs variance 

**Accuracy**
* Random Forest and Extra Trees outperform Decision Trees

**Variance**
* Decision trees → high
* Random Forest → medium
* Extra Trees → low

Notes:  
bagging helps decrease variance and reduce overfitting of base models  
boosting helps decrease bias by correcting errors iteratively  
stacking improves overall error by combining complementary model families

### Out-of-Bag Evaluation

* For each model, there are instances that are not trained on (because of the replacement). They are called out-of-bag instances (oob).
* oobs are used for evaluation without the need for a separate validation set (out of bag error)

Notes:  
As every single weak learner does not use *all* observations --> no additional validation set needed for evaluation.  
However, extra trees use the entire dataset. Here we cannot evaluate out of bag.  

This approach does not replace train test split but is used for hyperparameter tuning.

<div class="slide-chapter">

## II. Boosting
    
</div>

### Boosting

<div class="group">
    <div class="text">

Boosting builds a strong model by adding weak learners step by step.

Where:

* each new learner focuses on errors made so far
* later learners are trained on a reweighted or residual-focused view of the data
* learners are added **sequentially**, not independently
* decision trees are common base learners

AIPM intuition: think of iterative error-correction, where each round targets what the previous round missed.
    </div>
    <div class="images">
        <img src="../../images/ensemble_methods/boosting_step.png" width="750">
    </div>
</div> 

### Boosting algorithm

INIT
* Start with equal weight for all training instances.

LOOP
* Increase attention on previously mispredicted cases (or residuals).
* Train the next learner.
* Combine all learners into one weighted prediction.

### Boosting

<div class="group">
    <div class="text">

We do not go back and adjust earlier parameters → method is called **forward stagewise additive modeling**
    </div>
    <div class="images">
        <img src="../../images/ensemble_methods/boosting.png">
    </div>
</div>

Notes:  
Compared with bagging, boosting does not train all models independently.  
Instead, each new learner focuses on what previous learners missed.  
Instance weighting and learner weighting are related but distinct concepts.

### Well-known Boosting Algorithms

| Algorithm | Key Characteristics | Pros | Cons |
|-----------|---------------------|------|------|
| AdaBoost | - Focuses on misclassified samples<br>- Assigns weights to weak learners | - Simple and intuitive<br>- Effective for binary classification<br>- Less prone to overfitting | - Sensitive to noisy data and outliers<br>- Can be computationally expensive<br> |
| Gradient Boosting | - Instead of weighing difficult datapoint, GB cares for the *residuals*<br>- Uses gradient descent for minimization<br>- Builds trees sequentially | - Flexible and versatile<br>- Handles different loss functions<br>- Good performance on various tasks | - Can be slow to train<br>- Prone to overfitting if not tuned properly<br>- Requires careful parameter tuning |
| XGBoost | - Regularized form of Gradient Boosting<br>- Uses second-order gradients<br>- most widely used | - Fast and efficient (parallelization)<br>- Handles missing values well<br>- Built-in regularization | - Can be memory-intensive<br>- May overfit on small datasets<br>- Complex to tune |
| LightGBM | - Uses histogram-based algorithms<br>- Leaf-wise tree growth | - Fast training and prediction<br>- Low memory usage<br>- Handles large datasets efficiently | - May overfit on small datasets<br>- Sensitive to overfitting if not configured properly<br>- Less interpretable than simpler models |
| CatBoost | - Handles categorical features automatically<br>- Uses ordered boosting | - Excellent performance on categorical data<br>- Good default parameters<br>- Less prone to overfitting | - Can be slower than other algorithms<br>- May underperform on purely numerical data<br>- Limited hyperparameter tuning options |


Notes:  
In boosting, difficult instances often get higher importance in later rounds.  
How learner outputs are weighted depends on the algorithm:
- AdaBoost weights learners by individual performance.
- Gradient Boosting adds learners to reduce residual error step by step.

Choose among algorithms based on data type, compute budget, and tuning capacity.

<div class="slide-chapter">

## III. Stacking
    
</div>


### Stacking


<div class="group">
    <div class="text_70">
combining different  
models 
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../../images/ensemble_methods/stacking.png">
    </div>
</div> 




Notes:  
- Base models generate predictions from training data.
- These predictions become new features for a final model (meta-learner).
- To avoid leakage, do not train the meta-learner on predictions from the same rows used to train base models.
- Use either:
  - a holdout set (blending), or
  - out-of-fold predictions from cross-validation.

## Ensemble Learning: Leveraging Multiple Perspectives

<div class="group">
    <div class="text_70">

A collection of models working together to produce a single, stronger prediction.

**Key Strategies:**
* **<span class="color-brand">Bagging</span>**: trains independent models on random subsets to reduce variance.
* **<span class="color-brand">Boosting</span>**: trains sequentially to correct errors and reduce bias.
* **<span class="color-brand">Stacking</span>**: combines different model families with a meta-learner.

    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../../images/ensemble_methods/more_opinions.png">
    </div>
</div>

## Ensemble Learning: Leveraging Multiple Perspectives

<div class="group">
    <div class="text_70">

**AIPM selection guide:**
* Use **Bagging / Random Forest** for fast, strong tabular baselines with lower overfitting risk.
* Use **Extra Trees** when you want speed and strong variance reduction.
* Use **Boosting** when top predictive performance matters and tuning time is available.
* Use **Stacking** when you already have several good but different models.

**Core advantages:**
* better robustness
* better generalization
* less dependence on one model's blind spots
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../../images/ensemble_methods/more_opinions.png">
    </div>
</div>

Notes:  
This overview slide helps students reset the mental map after finishing bagging and before moving into boosting.

## Quick Comparison: Which Ensemble Should I Use?

| Technique | Best when | Strengths | Trade-offs |
|---|---|---|---|
| Bagging / Random Forest | You need a strong baseline fast on tabular data | Stable, robust, low tuning effort, handles nonlinearity | Less interpretable than one tree; can be large |
| Extra Trees | You want faster training and lower variance | Very fast, strong variance reduction, good default behavior | Slightly higher bias than Random Forest in some cases |
| Gradient Boosting family | You need top predictive performance | Often best accuracy on structured data; flexible objectives | More tuning, longer training, easier to overfit |
| XGBoost / LightGBM / CatBoost | You have medium-to-large data and care about leaderboard-level quality | Efficient implementations, advanced regularization, strong practical performance | Tool choice and tuning complexity increase |
| Stacking | You already have several diverse good models | Can improve performance by combining complementary strengths | Higher complexity, stricter validation setup, harder to debug |

AIPM rule of thumb:

1. Start with Random Forest as your baseline.
2. Move to boosting when baseline quality is not enough.
3. Use stacking only when the extra complexity is justified by business impact.

<div class="slide-chapter">

## 4. Classifier calibration

</div>

<div class="alert alert-block alert-info">
<b>Note:</b> 
    
Methods based on the values of the probabilities need to return calibrated probabilities.

</div>

Notes:  
Classifiers output probabilities, but those probabilities are often not perfectly calibrated.  
Calibration maps raw model confidence to probabilities that better match observed outcomes.

Core idea: for a well-calibrated model, predictions around a given confidence level should be correct at about that same frequency.

### Calibrated probabilities

<div class="group">
    <div class="text">
        
Fitting a regressor (called a calibrator) that maps the output of the classifier (decision_function or predict_proba) to a calibrated probability in [0, 1]

[sklearn - calibrated probabilities](https://scikit-learn.org/stable/modules/calibration.html)
    </div>
    <div class="images">
        <img src="../../images/ensemble_methods/img_p56_3_1.png">
    </div>
</div>

Notes:  
Calibration workflow:
1. Collect predicted probabilities and true labels.
2. Group predictions into probability bins.
3. Compare predicted confidence with observed positive rate in each bin.
4. Fit a calibrator to map raw probabilities to calibrated probabilities.

### Frequency of predicted probabilities

<div class="group">
    <div class="text">
        
confidence and falsehood vs. shyness and correctness

*example*
    </div>
    <div class="images">
        <img src="../../images/ensemble_methods/img_p56_3_2.png">
    </div>
</div>

Notes:  
This chart shows how often each classifier predicts different confidence levels.  
It is related to model behavior, not calibration quality by itself.

- U-shaped curves: more confident predictions.
- Mountain-shaped curves: more cautious predictions around 50-50.

<div class="slide-chapter">

## 5. Interpretability
    
</div>

### Interpreting black box models (like ensemble models)
* Linear models are popular due to interpretability, but poor performance
* Conclusions on the importance of a variable should be based on models with good accuracy
* Some of the approaches for measuring the effect of variables:
    * partial dependence plot
    * permutation feature importance


Notes:  
Why discuss interpretability here? Ensemble methods are often black-box models, but we still need feature-level insight for product decisions.  
Two practical tools shown here are partial dependence plots (PDP) and permutation feature importance (PFI).

### Partial dependence plots - Pros

<div class="group">
    <div class="text">
        
* Intuitive: the partial dependence function at a particular feature value represents the average prediction if we force all data points to assume that value
* causal interpretation (for the model)

    </div>
    <div class="images">
        <img src="../../images/ensemble_methods/img_p59_1.png">
    </div>
</div>

Notes:  
1. Train the model.
2. Set one feature to a fixed value across all rows.
3. Predict and average the output.
4. Repeat this across many values of that feature.
5. Repeat for other features.

This reveals how model predictions change as a feature value changes, while averaging over the rest.

### Partial dependence plots - Cons

<div class="group">
    <div class="text">

* maximum 2 features per plot
* assumption of independence between features
* Some PDPs omit showing feature value distribution, risking misinterpretation.

    </div>
    <div class="images">
        <img src="../../images/ensemble_methods/img_p59_1.png">
    </div>
</div>

### Permutation feature importance - Pros

<div class="group">
    <div class="text">

* nice interpretation: the loss if the feature is removed
* it takes into account the interaction with other features
* does not require retraining the model

    </div>
    <div class="images">
        <img src="../../images/ensemble_methods/img_p60_1.png">
    </div>
</div>

Notes:  
1. Evaluate baseline model performance (for example, accuracy or MSE).
2. Shuffle one feature column.
3. Re-evaluate performance and measure the drop.
4. Repeat several times for stability.

Larger performance drops indicate more important features.

### Permutation feature importance - Cons

<div class="group">
    <div class="text">

* linked to the error of model
* doesn’t deal well with correlated features
* lacks directional insight

    </div>
    <div class="images">
        <img src="../../images/ensemble_methods/img_p60_1.png">
    </div>
</div>

Notes:  
"Linked to model error": if the model underfits or overfits, PFI can be misleading.  
"Lacks directional insight": PFI shows importance magnitude, but not whether increasing a feature pushes predictions up or down.

## References

* Machine learning a probabilistic perspective - Kevin p. Murphy
* https://arxiv.org/pdf/1106.0257.pdf
* https://arxiv.org/pdf/1307.6522.pdf
* [Voting classifier blogpost](https://medium.com/@sanchitamangale12/voting-classifier-1be10db6d7a5)
* [Difference between extra trees and random forest](https://quantdare.com/what-is-the-difference-between-extra-trees-and-random-fo)
* [Explaining Adaboost, also including bagging and stacking](https://www.digitalocean.com/community/tutorials/adaboost-optimizer)
* [Videos about calibration](https://classifier-calibration.github.io/videos/)  
* [Calibration with sklearn](https://machinelearningmastery.com/calibrated-classification-model-in-scikit-learn/)  
* [Interpretable ML book](https://christophm.github.io/interpretable-ml-book)
* [Partial dependency plots](https://www.kaggle.com/dansbecker/partial-dependence-plots)
* &rarr; [SHapley Additive exPlanations](https://shap.readthedocs.io/en/latest/) &larr;